This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install keras keras-hub --upgrade -q

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

In [ ]:
# @title
import os
from IPython.core.magic import register_cell_magic

@register_cell_magic
def backend(line, cell):
    current, required = os.environ.get("KERAS_BACKEND", ""), line.split()[-1]
    if current == required:
        get_ipython().run_cell(cell)
    else:
        print(
            f"This cell requires the {required} backend. To run it, change KERAS_BACKEND to "
            f"\"{required}\" at the top of the notebook, restart the runtime, and rerun the notebook."
        )

## Best practices for the real world

### Getting the most out of your models

#### Hyperparameter optimization

##### Using KerasTuner

In [1]:
!pip install keras-tuner -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.1 MB/s eta 0:00:00


In [2]:
import keras
from keras import layers

def build_model(hp):
    units = hp.Int(name="units", min_value=16, max_value=128, step=16)
    dropout_rate = hp.Float("dropout", 0.0, 0.5, step=0.1)
    model = keras.Sequential(
        [
            layers.Dense(units, activation="relu"),
            layers.Dropout(dropout_rate),
            layers.Dense(10, activation="softmax"),
        ]
    )
    optimizer = hp.Choice(name="optimizer", values=["rmsprop", "adam", "sgd"])
    # use_batch_norm = hp.Boolean("use_batchnorm")
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    print(f"units={units}, dropout={dropout_rate}, optimizer={optimizer}")
    return model

In [ ]:
# import keras_tuner as kt

# class SimpleMLP(kt.HyperModel):
#     def __init__(self, num_classes):
#         self.num_classes = num_classes

#     def build(self, hp):
#         units = hp.Int(name="units", min_value=16, max_value=64, step=16)
#         model = keras.Sequential(
#             [
#                 layers.Dense(units, activation="relu"),
#                 layers.Dense(self.num_classes, activation="softmax"),
#             ]
#         )
#         optimizer = hp.Choice(name="optimizer", values=["rmsprop", "adam"])
#         model.compile(
#             optimizer=optimizer,
#             loss="sparse_categorical_crossentropy",
#             metrics=["accuracy"],
#         )
#         return model

# hypermodel = SimpleMLP(num_classes=10)

In [5]:
import keras_tuner as kt
tuner = kt.BayesianOptimization(
    build_model,
    objective="val_accuracy",
    max_trials=5,
    # каждая комбинация гиперпараметров будет обучена 2 раза
    # (с разными случайными инициализациями весов), а результат усреднен
    executions_per_trial=1,
    directory="mnist_kt_test",
    # Если в процессе поиска произошел сбой, вы всегда сможете
    # перезапустить его — просто укажите в настройках тюнера overwrite=False,
    # чтобы он смог возобновить перебор настроек, опираясь на сохраненные на диске журналы.
    overwrite=True,
)

units=16, dropout=0.0, optimizer=rmsprop


In [6]:
tuner.search_space_summary()

Search space summary
Default search space size: 3
units (Int)
{'default': None, 'conditions': [], 'min_value': 16, 'max_value': 128, 'step': 16, 'sampling': 'linear'}
dropout (Float)
{'default': 0.0, 'conditions': [], 'min_value': 0.0, 'max_value': 0.5, 'step': 0.1, 'sampling': 'linear'}
optimizer (Choice)
{'default': 'rmsprop', 'conditions': [], 'values': ['rmsprop', 'adam', 'sgd'], 'ordered': False}


In [7]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.reshape((-1, 28 * 28)).astype("float32") / 255
x_test = x_test.reshape((-1, 28 * 28)).astype("float32") / 255
x_train_full = x_train[:]
y_train_full = y_train[:]
num_val_samples = 10000
x_train, x_val = x_train[:-num_val_samples], x_train[-num_val_samples:]
y_train, y_val = y_train[:-num_val_samples], y_train[-num_val_samples:]
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5),
]
tuner.search(
    x_train,
    y_train,
    batch_size=128,
    epochs=100,
    validation_data=(x_val, y_val),
    callbacks=callbacks,
    verbose=2,
)

Trial 5 Complete [00h 00m 49s]
val_accuracy: 0.9781000018119812

Best val_accuracy So Far: 0.9781000018119812
Total elapsed time: 00h 05m 58s


In [15]:
import pandas as pd

trials = tuner.oracle.get_best_trials(num_trials=10)

data = []
for i, trial in enumerate(trials, 1):
    data.append({
        'Rank': i,
        'Trial ID': trial.trial_id,
        'Val Accuracy': f"{trial.score:.4f}",
        'Units': trial.hyperparameters.get('units'),
        'Dropout': trial.hyperparameters.get('dropout'),
        'Optimizer': trial.hyperparameters.get('optimizer')
    })

df = pd.DataFrame(data)
df

,Rank,Trial ID,Val Accuracy,Units,Dropout,Optimizer
0,1,4,0.9781,112,0.0,adam
1,2,3,0.9773,112,0.4,adam
2,3,0,0.9755,80,0.1,rmsprop
3,4,1,0.9665,32,0.0,rmsprop
4,5,2,0.9664,64,0.0,sgd


In [9]:
# top_n = 4
# best_hps = tuner.get_best_hyperparameters(top_n)

In [10]:
# def get_best_epoch(hp):
#     model = build_model(hp)
#     callbacks = [
#         keras.callbacks.EarlyStopping(
#             monitor="val_loss", mode="min", patience=10
#         )
#     ]
#     history = model.fit(
#         x_train,
#         y_train,
#         validation_data=(x_val, y_val),
#         epochs=100,
#         batch_size=128,
#         callbacks=callbacks,
#     )
#     val_loss_per_epoch = history.history["val_loss"]
#     best_epoch = val_loss_per_epoch.index(min(val_loss_per_epoch)) + 1
#     print(f"Best epoch: {best_epoch}")
#     return best_epoch

In [11]:
# def get_best_trained_model(hp):
#     best_epoch = get_best_epoch(hp)
#     model = build_model(hp)
#     model.fit(
#         x_train_full, y_train_full,     #на ВСЕХ данных
#         batch_size=128,
#         epochs=int(best_epoch * 1.2)
#     )
#     return model

# best_models = []
# for hp in best_hps:
#     model = get_best_trained_model(hp)
#     model.evaluate(x_test, y_test)
#     best_models.append(model)

In [16]:
# Если вас не беспокоит некоторое отставание в качестве модели,
# можно пойти по более короткому пути: просто используйте тюнер
# для загрузки наиболее эффективной модели с лучшими весами,
# сохраненными во время поиска гиперпараметров,
# без повторного обучения новых моделей с нуля:
# "Отставание" - значит модель обучалась:
#  - только на трейне (без +вала)
#  - при меньшем числе эпох (без 1.2)
top_n = 4
best_models = tuner.get_best_models(top_n)

units=112, dropout=0.0, optimizer=adam


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


units=112, dropout=0.4, optimizer=adam
units=80, dropout=0.1, optimizer=rmsprop


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


units=32, dropout=0.0, optimizer=rmsprop


In [23]:
best_models[0]

<Sequential name=sequential, built=True>

##### The art of crafting the right search space

##### The future of hyperparameter tuning: automated machine learning

#### Model ensembling

### Scaling up model training with multiple devices

#### Multi-GPU training

##### Data parallelism: Replicating your model on each GPU

##### Model parallelism: Splitting your model across multiple GPUs

#### Distributed training in practice

##### Getting your hands on two or more GPUs

##### Using data parallelism with JAX

##### Using model parallelism with JAX

###### The DeviceMesh API

###### The LayoutMap API

#### TPU training

##### Using step fusing to improve TPU utilization

### Speeding up training and inference with lower-precision computation

##### Understanding floating-point precision

##### Float16 inference

##### Mixed-precision training

##### Using loss scaling with mixed precision

##### Beyond mixed precision: float8 training

#### Faster inference with quantization

In [ ]:
from keras import ops

x = ops.array([[0.1, 0.9], [1.2, -0.8]])
kernel = ops.array([[-0.1, -2.2], [1.1, 0.7]])

In [ ]:
def abs_max_quantize(value):
    abs_max = ops.max(ops.abs(value), keepdims=True)
    scale = ops.divide(127, abs_max + 1e-7)
    scaled_value = value * scale
    scaled_value = ops.clip(ops.round(scaled_value), -127, 127)
    scaled_value = ops.cast(scaled_value, dtype="int8")
    return scaled_value, scale

int_x, x_scale = abs_max_quantize(x)
int_kernel, kernel_scale = abs_max_quantize(kernel)

In [ ]:
int_y = ops.matmul(int_x, int_kernel)
y = ops.cast(int_y, dtype="float32") / (x_scale * kernel_scale)

In [ ]:
y

In [ ]:
ops.matmul(x, kernel)